# Texas RRC Refinery Analysis

Refinery-agnostic analysis notebook.  
Set `REFINERY_ID` in the **Config** cell below and run all cells.

In [64]:
# ── Config ─────────────────────────────────────────────────────────────────
REFINERY_ID   = '03-0052'           # ← change to any refinery ID
DATE_FROM     = '2020-01'           # 'YYYY-MM' inclusive lower bound  (None = all)
DATE_TO       = None                # 'YYYY-MM' inclusive upper bound  (None = all)

CLEANED_DIR   = 'data/cleaned'      # folder where curated CSVs live
# ---------------------------------------------------------------------------

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Load data ──────────────────────────────────────────────────────────────
csv_path = Path(CLEANED_DIR) / f'{REFINERY_ID}.csv'
assert csv_path.exists(), (
    f'No cleaned CSV found for {REFINERY_ID}.  '
    f'Expected: {csv_path}\n'
    f'Available: {[p.stem for p in Path(CLEANED_DIR).glob("*.csv")]}'
)

raw = pd.read_csv(csv_path, dtype={'id': str})
raw['date'] = pd.to_datetime(raw['date'], dayfirst=True)
raw['period'] = raw['date'].dt.to_period('M')

# Date bounds
if DATE_FROM:
    raw = raw[raw['date'] >= pd.Timestamp(DATE_FROM + '-01')]
if DATE_TO:
    raw = raw[raw['date'] <= pd.to_datetime(DATE_TO + '-01') + pd.offsets.MonthEnd(0)]

# Normalise the boolean column (CSV may store TRUE/FALSE as strings)
raw['balanced'] = raw['balanced'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False, '1': True, '0': False})

# Infer refinery name from data if available, else use ID
REFINERY_NAME = REFINERY_ID

numeric_cols = ['Storage Beginning of Month', 'Receipts', 'Input',
                'Products Manufactured', 'Fuel Used', 'Deliveries',
                'Storage End of Month']

print(f'Loaded {len(raw):,} rows for refinery {REFINERY_NAME}')
print(f'Period: {raw["date"].min().strftime("%b %Y")} → {raw["date"].max().strftime("%b %Y")}')
print(f'Products: {raw["material"].nunique()} | Operators: {raw["id"].nunique()}')
raw.head()

Loaded 1,632 rows for refinery 03-0052
Period: Jan 2020 → Aug 2025
Products: 24 | Operators: 1


,Code,Storage Beginning of Month,Receipts,Input,Products Manufactured,Fuel Used,Deliveries,Storage End of Month,id,date,extraction_mode,extraction_tier,balance_diff,balanced,material,period
0,1,717,5926,5926,413329,0,413238,808,03-0052,2020-10-31,claude_vision,1,0,True,Propane,2020-10
1,2,43569,571821,556096,183203,0,192998,49499,03-0052,2020-10-31,claude_vision,1,0,True,Butane,2020-10
2,3,0,0,0,0,0,0,0,03-0052,2020-10-31,claude_vision,1,0,True,Butane-Propane,2020-10
3,4,2136733,6712,2102,8542191,0,8659680,2023854,03-0052,2020-10-31,claude_vision,1,0,True,Motor Gasoline,2020-10
4,5,0,0,0,0,0,0,0,03-0052,2020-10-31,claude_vision,1,0,True,Kerosene,2020-10


## 1  Data Quality

In [65]:
# ── 1a  Balance check heat-map ──────────────────────────────────────────────
pivot_bal = raw.pivot_table(index='material', columns='period',
                            values='balanced', aggfunc='min')
pivot_bal = pivot_bal.astype(float)   # True→1, False→0, NaN→NaN
pivot_bal.columns = pivot_bal.columns.astype(str)   # Period → str so Plotly can serialize

fig = px.imshow(
    pivot_bal,
    color_continuous_scale=['red', 'green'],
    aspect='auto',
    title=f'{REFINERY_NAME} — Balance Status by Product & Month',
    labels=dict(color='Balanced'),
    zmin=0, zmax=1,
)
fig.update_xaxes(tickangle=45)
fig.show()

# ── 1d  Balance summary table ───────────────────────────────────────────────
active = raw[raw[numeric_cols].sum(axis=1) > 0].copy()
n_unbal = (~active['balanced']).sum()
pct_unbal = n_unbal / len(active) * 100 if len(active) else 0

print(f'Total rows        : {len(raw):,}')
print(f'Active rows       : {len(active):,}')
print(f'Unbalanced active : {n_unbal} ({pct_unbal:.1f}%)')

if n_unbal:
    print('\nUnbalanced records:')
    display(
        active[~active['balanced']][['date','material','balance_diff'] + numeric_cols]
        .sort_values('balance_diff', key=abs, ascending=False)
    )   

Total rows        : 1,632
Active rows       : 563
Unbalanced active : 9 (1.6%)

Unbalanced records:


,date,material,balance_diff,Storage Beginning of Month,Receipts,Input,Products Manufactured,Fuel Used,Deliveries,Storage End of Month
620,2025-04-30,Special Naphthas,-330,172368,0,1161,0,0,-165,171372
1008,2021-02-28,Propane,0,485,0,0,0,0,0,885
1009,2021-02-28,Butane,0,39754,0,0,0,0,0,37064
1011,2021-02-28,Motor Gasoline,0,1967840,0,0,0,0,0,1778898
1015,2021-02-28,Other Middle Distillates,0,532089,0,0,0,0,0,336573
1020,2021-02-28,"Fuel Oils #5,#6 For Utility Use",0,558908,0,0,0,0,0,575556
1025,2021-02-28,Other Residual Fuels,0,156953,0,0,0,0,0,104354
1030,2021-02-28,Miscellaneous,0,1776479,0,0,0,0,0,1688222
1111,2021-06-30,Other Middle Distillates,0,708779,0,30374,3616416,0,3580365,714456


In [66]:
# ── 1b  Carry check: End of Month T == Beginning of Month T+1 ───────────────
# Sort per product chronologically so shift() steps month-by-month
CARRY_TOL = 2   # barrels — same tolerance as the within-month balance check

chk = raw.sort_values(['material', 'date']).copy()
chk['prev_end']   = chk.groupby('material')['Storage End of Month'].shift(1)
chk['carry_diff'] = (chk['Storage Beginning of Month'] - chk['prev_end']).abs()
# First month per product has no previous → always OK
chk['carry_ok']   = chk['carry_diff'].lt(CARRY_TOL) | chk['prev_end'].isna()

# Heat-map: one cell per (product × month), green = carry OK, red = mismatch
pivot_carry = chk.pivot_table(
    index='material', columns='period', values='carry_ok', aggfunc='min'
).astype(float)
pivot_carry.columns = pivot_carry.columns.astype(str)   # Period → str so Plotly can serialize

fig = px.imshow(
    pivot_carry,
    color_continuous_scale=['red', 'green'],
    aspect='auto',
    title=f'{REFINERY_NAME} — Carry Consistency (End of T = Beginning of T+1)',
    labels=dict(color='Carry OK'),
    zmin=0, zmax=1,
)
fig.update_xaxes(tickangle=45)
fig.show()

# ── 1c  Flagged carry mismatches ─────────────────────────────────────────────
carry_fail = chk[~chk['carry_ok']].copy()
n_carry_fail = len(carry_fail)
print(f'Carry mismatches : {n_carry_fail}  (out of {len(chk) - chk["prev_end"].isna().sum()} comparable pairs)')

if n_carry_fail:
    display(
        carry_fail[['date', 'material', 'prev_end', 'Storage Beginning of Month', 'carry_diff']]
        .rename(columns={
            'prev_end'                      : 'Prev Month End Storage',
            'Storage Beginning of Month'    : 'This Month Begin Storage',
            'carry_diff'                    : 'Discrepancy (bbls)',
        })
        .sort_values('Discrepancy (bbls)', ascending=False)
        .reset_index(drop=True)
    )

# ── 1d  Combined quality summary per product ─────────────────────────────────
# One row per product: within-month balance pass-rate + cross-month carry pass-rate
qual = (
    chk.groupby('material')
    .apply(lambda g: pd.Series({
        'months_active'  : int((g[numeric_cols].sum(axis=1) > 0).sum()),
        'balance_ok_pct' : round(g.loc[g[numeric_cols].sum(axis=1) > 0, 'balanced'].mean() * 100, 1)
                          if (g[numeric_cols].sum(axis=1) > 0).any() else float('nan'),
        'carry_ok_pct'   : round(g.loc[~g['prev_end'].isna(), 'carry_ok'].mean() * 100, 1)
                          if (~g['prev_end'].isna()).any() else float('nan'),
        'carry_fails'    : int((~g['carry_ok'] & ~g['prev_end'].isna()).sum()),
    }), include_groups=False)
    .sort_values('carry_ok_pct')
)
print('\nPer-product quality (sorted by worst carry consistency):')
display(qual)   

Carry mismatches : 3  (out of 1608 comparable pairs)


,date,material,Prev Month End Storage,This Month Begin Storage,Discrepancy (bbls)
0,2025-07-31,Miscellaneous,1540176.0,1540275,99.0
1,2021-04-30,Miscellaneous,2420593.0,2420591,2.0
2,2022-07-31,Miscellaneous,2115227.0,2115225,2.0



Per-product quality (sorted by worst carry consistency):


,months_active,balance_ok_pct,carry_ok_pct,carry_fails
material,,,,
Miscellaneous,68.0,98.5,95.5,3.0
Aviation Gasoline,0.0,NaN,100.0,0.0
Butane,68.0,98.5,100.0,0.0
Bunker C,0.0,NaN,100.0,0.0
Crude Oil,68.0,100.0,100.0,0.0
Diesel Fuel,0.0,NaN,100.0,0.0
Fuel Oils #4 For Non-Utility Use,0.0,NaN,100.0,0.0
Butane-Propane,0.0,NaN,100.0,0.0
Fuel Oils #4 For Utility Use,0.0,NaN,100.0,0.0


## 2  Crude Throughput (Input)

In [58]:
# ── Detect crude / charge codes ─────────────────────────────────────────────
# Code 1 = Crude; also include any product with significant 'Input' volume
crude_codes = raw.groupby('material')['Input'].sum()
crude_codes = crude_codes[crude_codes > crude_codes.max() * 0.05].index.tolist()

crude = raw[raw['material'].isin(crude_codes)].copy()
crude_monthly = crude.groupby('period')[['Input', 'Receipts', 'Products Manufactured']].sum().reset_index()
crude_monthly['period_dt'] = crude_monthly['period'].dt.to_timestamp()

fig = go.Figure()
fig.add_trace(go.Bar(x=crude_monthly['period_dt'], y=crude_monthly['Input'],
                     name='Input (crude charge)', marker_color='steelblue'))
fig.add_trace(go.Scatter(x=crude_monthly['period_dt'],
                         y=crude_monthly['Input'].rolling(3, center=True).mean(),
                         mode='lines', name='3-mo rolling avg',
                         line=dict(color='orange', width=2)))
fig.update_layout(
    title=f'{REFINERY_NAME} — Monthly Crude / Charge Input (Bbls)',
    xaxis_title='Month', yaxis_title='Barrels',
    legend=dict(orientation='h'),
    hovermode='x unified',
)
fig.show()

print(f'Avg monthly input : {crude_monthly["Input"].mean():,.0f} bbls')
print(f'Peak month        : {crude_monthly.loc[crude_monthly["Input"].idxmax(), "period_dt"].strftime("%b %Y")} '
      f'({crude_monthly["Input"].max():,.0f} bbls)')

Avg monthly input : 6,906,394 bbls
Peak month        : Dec 2021 (8,286,522 bbls)


## 3  Product Output Mix

In [59]:
# ── 3a  Total deliveries by product (pie) ──────────────────────────────────
prod_totals = (
    raw.groupby('material')['Deliveries'].sum()
    .sort_values(ascending=False)
    .reset_index()
)
prod_totals = prod_totals[prod_totals['Deliveries'] > 0]

fig = px.pie(prod_totals, values='Deliveries', names='material',
             title=f'{REFINERY_NAME} — Total Deliveries by Product',
             hole=0.35)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

# ── 3b  Monthly deliveries by top products (stacked area) ──────────────────
TOP_N = 15
top_products = prod_totals['material'].head(TOP_N).tolist()

monthly_prod = (
    raw[raw['material'].isin(top_products)]
    .groupby(['period', 'material'])['Deliveries']
    .sum()
    .reset_index()
)
monthly_prod['period_dt'] = monthly_prod['period'].dt.to_timestamp()

fig = px.area(monthly_prod, x='period_dt', y='Deliveries', color='material',
              title=f'{REFINERY_NAME} — Monthly Deliveries (Top {TOP_N} Products)',
              labels={'period_dt': 'Month', 'Deliveries': 'Barrels'},
              )
fig.update_layout(hovermode='x unified', legend_title='Product')
fig.show()

In [60]:
# ── 3a  Total deliveries by product (pie) ──────────────────────────────────
prod_totals = (
    raw.groupby('material')['Products Manufactured'].sum()
    .sort_values(ascending=False)
    .reset_index()
)
prod_totals = prod_totals[prod_totals['Products Manufactured'] > 0]

fig = px.pie(prod_totals, values='Products Manufactured', names='material',
             title=f'{REFINERY_NAME} — Total Products Manufactured by Product',
             hole=0.35)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

# ── 3b  Monthly deliveries by top products (stacked area) ──────────────────
TOP_N = 15
top_products = prod_totals['material'].head(TOP_N).tolist()

monthly_prod = (
    raw[raw['material'].isin(top_products)]
    .groupby(['period', 'material'])['Products Manufactured']
    .sum()
    .reset_index()
)
monthly_prod['period_dt'] = monthly_prod['period'].dt.to_timestamp()

fig = px.area(monthly_prod, x='period_dt', y='Products Manufactured', color='material',
              title=f'{REFINERY_NAME} — Monthly Products Manufactured (Top {TOP_N} Products)',
              labels={'period_dt': 'Month', 'Products Manufactured': 'Barrels'},
              )
fig.update_layout(hovermode='x unified', legend_title='Product')
fig.show()

## 4  Inventory & Deliveries

In [61]:
# ── 4a  Total inventory trend (all products combined) ──────────────────────
inv_monthly = (
    raw.groupby('period')[['Storage Beginning of Month', 'Storage End of Month',
                            'Receipts', 'Deliveries']].sum().reset_index()
)
inv_monthly['period_dt'] = inv_monthly['period'].dt.to_timestamp()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=inv_monthly['period_dt'],
                         y=inv_monthly['Storage End of Month'],
                         name='End-of-Month Storage', fill='tozeroy',
                         line=dict(color='royalblue')), secondary_y=False)
fig.add_trace(go.Bar(x=inv_monthly['period_dt'],
                     y=inv_monthly['Deliveries'],
                     name='Deliveries', opacity=0.6,
                     marker_color='salmon'), secondary_y=True)
fig.update_layout(title=f'{REFINERY_NAME} — Inventory & Deliveries',
                  hovermode='x unified', legend=dict(orientation='h'))
fig.update_yaxes(title_text='Storage (Bbls)', secondary_y=False)
fig.update_yaxes(title_text='Deliveries (Bbls)', secondary_y=True)
fig.show()

# ── 4b  Inventory by product (end-of-month, latest period) ────────────────
latest = raw[raw['period'] == raw['period'].max()]
inv_latest = latest.groupby('material')['Storage End of Month'].sum().sort_values(ascending=False)
inv_latest = inv_latest[inv_latest > 0]

fig = px.bar(inv_latest.reset_index(),
             x='material', y='Storage End of Month',
             title=f'{REFINERY_NAME} — End-of-Month Storage ({raw["period"].max()})',
             labels={'Storage End of Month': 'Barrels', 'material': 'Product'})
fig.update_xaxes(tickangle=45)
fig.show()

## 5  Yield Ratios

In [62]:
# Yield = Products Manufactured / Input  (only months where Input > 0)
yield_df = (
    raw.groupby('period')[['Products Manufactured', 'Input']].sum().reset_index()
)
yield_df = yield_df[yield_df['Input'] > 0].copy()
yield_df['yield_pct'] = yield_df['Products Manufactured'] / yield_df['Input'] * 100
yield_df['period_dt'] = yield_df['period'].dt.to_timestamp()

fig = go.Figure()
fig.add_trace(go.Scatter(x=yield_df['period_dt'], y=yield_df['yield_pct'],
                         mode='lines+markers', name='Yield %',
                         line=dict(color='mediumseagreen', width=2)))
fig.add_hline(y=yield_df['yield_pct'].mean(), line_dash='dash',
              annotation_text=f'Avg {yield_df["yield_pct"].mean():.1f}%',
              line_color='gray')
fig.update_layout(
    title=f'{REFINERY_NAME} — Monthly Conversion Yield (Products Mfg / Input)',
    xaxis_title='Month', yaxis_title='Yield %',
    hovermode='x unified',
)
fig.show()

# Per-product yield (manufactured vs input, aggregated over full period)
prod_yield = (
    raw.groupby('material')[['Products Manufactured', 'Input']].sum()
)
prod_yield = prod_yield[prod_yield['Input'] > 0].copy()
prod_yield['yield_pct'] = prod_yield['Products Manufactured'] / prod_yield['Input'] * 100
prod_yield = prod_yield[prod_yield['yield_pct'] > 0].sort_values('yield_pct', ascending=False)

if not prod_yield.empty:
    fig = px.bar(prod_yield.reset_index(),
                 x='material', y='yield_pct',
                 title=f'{REFINERY_NAME} — Per-Product Yield (full period)',
                 labels={'yield_pct': 'Yield %', 'material': 'Product'})
    fig.update_xaxes(tickangle=45)
    fig.show()

## 6  Operator / Period Comparison

In [63]:
# Split the dataset by operator (if the CSV covers multiple operators for the same ID)
# or by user-defined periods (e.g., pre- vs post-2022).

# ── 6a  Detect operator changes from extraction_mode or name column ─────────
# The cleaned CSV for 01-0692 spans multiple operators;
# approximate by labelling year-groups or using the 'id' column if it varies.

PERIOD_CUTS = {
    'Pre-2022'  : ('2020-01', '2021-12'),
    '2022-2023' : ('2022-01', '2023-12'),
    '2024+'     : ('2024-01', None),
}

def label_period(dt):
    for label, (start, end) in PERIOD_CUTS.items():
        s = pd.Timestamp(start)
        e = pd.Timestamp(end + '-01') + pd.offsets.MonthEnd(0) if end else pd.Timestamp('2099-12-31')
        if s <= dt <= e:
            return label
    return 'Other'

raw['op_period'] = raw['date'].apply(label_period)

# ── 6b  Key metrics by period ──────────────────────────────────────────────
period_summary = (
    raw.groupby('op_period')[['Input', 'Products Manufactured', 'Deliveries',
                               'Storage End of Month']]
    .mean()
    .round(0)
    .rename_axis('Period')
    .reindex([k for k in PERIOD_CUTS if k in raw['op_period'].unique()])
)
print('Average monthly volumes by operator period:')
display(period_summary)

# ── 6c  Stacked bar: Input vs Deliveries per period ────────────────────────
period_summary_reset = period_summary.reset_index()
fig = go.Figure()
for col, color in zip(['Input', 'Products Manufactured', 'Deliveries'],
                       ['steelblue', 'mediumseagreen', 'salmon']):
    fig.add_trace(go.Bar(name=col, x=period_summary_reset['Period'],
                         y=period_summary_reset[col], marker_color=color))
fig.update_layout(
    barmode='group',
    title=f'{REFINERY_NAME} — Avg Monthly Volumes by Period',
    yaxis_title='Barrels (avg / month)',
    legend=dict(orientation='h'),
)
fig.show()

Average monthly volumes by operator period:


,Input,Products Manufactured,Deliveries,Storage End of Month
Period,,,,
Pre-2022,289628.0,308458.0,368540.0,147463.0
2022-2023,301053.0,330381.0,391054.0,150355.0
2024+,266707.0,292632.0,346765.0,126502.0
